In [1]:
from ultralytics import YOLO
import cv2
import os

# Load YOLOv8 pretrained
model = YOLO("yolov8n.pt")

# Đọc ảnh
image_path = r"C:\Users\Admin\Desktop\dpl\Screenshot 2026-03-10 115447.png"
img = cv2.imread(image_path)

# Thư mục lưu ảnh
output_folder = "cropped_vehicles"
os.makedirs(output_folder, exist_ok=True)

# Detect
results = model(img)

vehicle_count = 0
CONF_THRESHOLD = 0.5
SHRINK_RATIO = 0.1   # thu nhỏ bounding box 10%

for r in results:
    for box in r.boxes:

        cls = int(box.cls[0])
        conf = float(box.conf[0])

        # Lọc phương tiện
        if cls in [2,3,5,7] and conf > CONF_THRESHOLD:

            x1,y1,x2,y2 = map(int, box.xyxy[0])

            # ===== Shrink bounding box =====
            w = x2 - x1
            h = y2 - y1

            x1 = int(x1 + w * SHRINK_RATIO)
            x2 = int(x2 - w * SHRINK_RATIO)
            y1 = int(y1 + h * SHRINK_RATIO)
            y2 = int(y2 - h * SHRINK_RATIO)

            # Đảm bảo không vượt ảnh
            x1 = max(0,x1)
            y1 = max(0,y1)
            x2 = min(img.shape[1],x2)
            y2 = min(img.shape[0],y2)

            # Crop
            vehicle_crop = img[y1:y2, x1:x2]

            # Lưu ảnh
            save_path = f"{output_folder}/vehicle_{vehicle_count}.jpg"
            cv2.imwrite(save_path, vehicle_crop)

            vehicle_count += 1

            # Vẽ bounding box
            cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.putText(img,f"Vehicle {conf:.2f}",
                        (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,(0,255,0),2)

# Hiển thị kết quả
cv2.imshow("Detected Vehicles", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

print(f"Đã crop {vehicle_count} phương tiện.")


0: 448x640 3 persons, 1 car, 38.5ms
Speed: 0.0ms preprocess, 38.5ms inference, 54.8ms postprocess per image at shape (1, 3, 448, 640)
Đã crop 1 phương tiện.
